# Put-Call Parity & Arbitrage Bounds

A from-scratch exploration of put-call parity, arbitrage bounds on option prices, American option
extensions, dividend adjustments, and synthetic position construction.

---

## The Most Elegant Equation in Finance

If someone asked you to distill the entire field of derivatives pricing into a single equation, it would be **put-call parity**. Not Black-Scholes (which requires assumptions about volatility and distributions), not the binomial model (which requires a tree structure), but this deceptively simple relationship:

$$\boxed{C - P = S - Ke^{-rT}}$$

where:
- $C$ = European call price
- $P$ = European put price (same strike and expiry)
- $S$ = current stock price
- $K$ = strike price
- $r$ = continuously compounded risk-free rate
- $T$ = time to expiry (in years)

> **In plain English:** A long call and a short put (at the same strike and expiry) behave exactly like owning the stock minus a risk-free bond that pays $K$ at expiry.

What makes this equation so remarkable? Three things:

1. **It requires NO assumptions about the stock price distribution.** The stock could follow geometric Brownian motion, jump-diffusion, stochastic volatility, or any other process. Parity still holds.
2. **It requires NO pricing model.** You do not need Black-Scholes, binomial trees, or Monte Carlo. It follows purely from the absence of arbitrage.
3. **It is exact, not approximate.** Unlike many financial relationships that hold "on average" or "in equilibrium," put-call parity holds to the penny for European options on the same underlying, strike, and expiry.

### A Brief History

Put-call parity was first formally described by Hans Stoll in his 1969 paper *"The Relationship Between Put and Call Option Prices."* However, the intuition goes back further -- option traders in the early 20th century already understood that calls and puts were two sides of the same coin. The relationship predates Black-Scholes (1973) by four years, which is telling: you can understand put-call parity without any stochastic calculus at all.

### Why It Works: The Replicating Portfolio Argument

The proof is elegant in its simplicity. We construct two portfolios and show they have identical payoffs at expiry, no matter what happens to the stock price. Since they always pay the same thing in the future, they must cost the same thing today (otherwise, arbitrage).

Consider two portfolios assembled today:

**Portfolio A ("Call + Cash"):** One European call option + cash of $Ke^{-rT}$ invested at the risk-free rate.

- The cash grows to exactly $K$ by expiry.
- If the stock ends above $K$: you exercise the call, pay $K$ (from your cash), and receive the stock worth $S_T$. Portfolio value = $S_T$.
- If the stock ends at or below $K$: the call expires worthless, and you keep the cash $K$. Portfolio value = $K$.

**Portfolio B ("Put + Stock"):** One European put option + one share of stock.

- If the stock ends above $K$: the put expires worthless, and you have the stock worth $S_T$. Portfolio value = $S_T$.
- If the stock ends at or below $K$: you exercise the put, sell the stock for $K$. Portfolio value = $K$.

Let us lay this out in a payoff table:

| Scenario | Portfolio A value | Portfolio B value |
|:---------|:------------------|:------------------|
| $S_T > K$ | $(S_T - K) + K = S_T$ | $0 + S_T = S_T$ |
| $S_T \leq K$ | $0 + K = K$ | $(K - S_T) + S_T = K$ |

Both portfolios produce $\max(S_T, K)$ at expiry. **Always. In every scenario. Without exception.**

By the **law of one price** (the foundation of no-arbitrage finance), two portfolios with identical future payoffs must have the same price today:

$$C + Ke^{-rT} = P + S$$

Rearranging gives us put-call parity: $C - P = S - Ke^{-rT}$.

> **Key Concept:** Put-call parity requires NO assumptions about the stock price distribution, volatility, or any pricing model. It is a pure no-arbitrage result. This makes it one of the most robust relationships in finance.

### Understanding the Equation Intuitively

Let us break down the equation $C - P = S - Ke^{-rT}$ piece by piece:

- **Left side: $C - P$** -- The cost of a "long call, short put" combination. This is sometimes called a **synthetic forward**. If you buy a call and sell a put at the same strike, you have effectively committed to buying the stock at price $K$ at expiry (the call gives you the right to buy; the short put gives you the obligation to buy).

- **Right side: $S - Ke^{-rT}$** -- The cost of buying the stock today and borrowing the present value of $K$. This is equivalent to entering a prepaid forward contract.

Both sides represent the same economic exposure: a commitment to own the stock at expiry, having effectively paid $K$ for it.

> **CFA Exam Tip:** A common exam question asks: "If a call is priced at $X$, what must the put be priced at?" Just rearrange parity: $P = C - S + Ke^{-rT}$. Plug in and compute. No Black-Scholes needed.

### Worked Example

$S = 100$, $K = 100$, $r = 5\%$, $T = 1$ year, $C = 12.50$

What should the put price be?

$$P = C - S + Ke^{-rT} = 12.50 - 100 + 100e^{-0.05} = 12.50 - 100 + 95.12 = 7.62$$

If the market put price is \$8.50 instead of \$7.62, an arbitrage opportunity exists! (We will explore exactly how to exploit this shortly.)

> **Common Mistake:** Forgetting that $Ke^{-rT}$ is the *present value* of $K$, not $K$ itself. For short-dated options (small $T$), the difference is small, but it matters for precision.

## 1. Motivation and Roadmap

**Put-call parity** is one of the most fundamental relationships in options pricing. It holds
regardless of the model used (BSM, binomial, etc.) and depends only on no-arbitrage.

It provides:
- A **consistency check** on option prices -- if your model violates parity, there is a bug.
- A recipe for constructing **arbitrage portfolios** when parity is violated in the market.
- The foundation for **synthetic positions** in hedging and market-making.
- A bridge between calls and puts -- if you can price one, you automatically know the other.

### What We Will Cover

This notebook proceeds in a logical sequence:

1. **Setup** -- Import libraries and define plotting conventions.
2. **European Put-Call Parity** -- Formal derivation and verification with Black-Scholes.
3. **Visualization** -- Graphical confirmation that both sides of the parity equation overlap.
4. **Arbitrage from Violations** -- Step-by-step construction of riskless profit when parity breaks.
5. **Lower Bounds on Option Prices** -- No-arbitrage constraints that every option must satisfy.
6. **American Options** -- Why parity becomes an inequality (the early exercise premium).
7. **Dividend Adjustments** -- How known dividends modify the parity relationship.
8. **Synthetic Positions** -- Building any position from the other three instruments.

Everything below is derived from first principles and implemented from scratch.### Why Put-Call Parity Matters Beyond Theory

Put-call parity isn't just an academic curiosity. It has immediate practical applications:

1. **Pricing checks:** Market makers verify their models satisfy parity before going live
2. **Implied borrowing rates:** Rearranging parity reveals the market's implied risk-free rate
3. **Dividend estimation:** Parity violations in index options reveal the market's expected dividend yield
4. **Regulatory arbitrage:** Some markets restrict short selling but not synthetic shorts via options


### Setup

Before we begin, let us set up our computational environment. We use the standard scientific Python stack:

- **NumPy** for array computations and vectorized arithmetic.
- **SciPy** for the cumulative normal distribution function (needed in Black-Scholes) and optimization.
- **Matplotlib** for all visualizations.

We also define some plotting constants for consistent color schemes throughout the notebook. The tolerance constants (`ATOL`, `RTOL`) are used to determine when numerical results are "close enough" to zero.

> **Key Concept:** We implement everything from scratch using NumPy and SciPy. No options pricing libraries are used. This is intentional -- understanding the mechanics is the point.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

With our environment ready, we can now move to the heart of the notebook: implementing and verifying put-call parity.

The cell above sets a random seed for reproducibility, configures figure defaults (10x6 inches, font size 12, light grid), and defines a consistent color palette: steelblue for calls/primary results, coral for puts/secondary results, seagreen for tertiary elements, and gold for accents like bonds.

---

## 3. European Put-Call Parity

### The Formal Relationship

For European options on a non-dividend-paying stock, put-call parity states:

$$\boxed{C - P = S_0 - K e^{-rT}}$$

Let us re-derive this carefully, paying attention to every step.

### Derivation via Replicating Portfolios

**Portfolio A**: Long one European call + invest $K e^{-rT}$ in a risk-free bond.

At expiry, the bond has grown to $K$, and the call pays $\max(S_T - K, 0)$. Together:

$$\text{Value of A at T} = \max(S_T - K, 0) + K$$

If $S_T > K$: this equals $(S_T - K) + K = S_T$.
If $S_T \leq K$: this equals $0 + K = K$.
So Portfolio A pays $\max(S_T, K)$.

**Portfolio B**: Long one European put + long one share of stock.

At expiry, the put pays $\max(K - S_T, 0)$, and the stock is worth $S_T$. Together:

$$\text{Value of B at T} = \max(K - S_T, 0) + S_T$$

If $S_T > K$: this equals $0 + S_T = S_T$.
If $S_T \leq K$: this equals $(K - S_T) + S_T = K$.
So Portfolio B also pays $\max(S_T, K)$.

Since both portfolios have the same payoff $\max(S_T, K)$ at time $T$ in every state of the world, **no-arbitrage** requires them to have the same value today:

$$C + K e^{-rT} = P + S_0$$

Rearranging: $C - P = S_0 - K e^{-rT}$.

### What Does Each Term Mean Economically?

| Term | Meaning |
|:-----|:--------|
| $C$ | The right (not obligation) to buy the stock at $K$ |
| $P$ | The right (not obligation) to sell the stock at $K$ |
| $S_0$ | The current market price of the stock |
| $Ke^{-rT}$ | Today's cost of guaranteeing $K$ dollars at time $T$ (a zero-coupon bond) |
| $C - P$ | Net cost of a "synthetic forward" at strike $K$ |
| $S_0 - Ke^{-rT}$ | Forward price minus strike, discounted (the prepaid forward value) |

> **Key Concept:** The term $Ke^{-rT}$ appears because money has time value. A dollar today is worth more than a dollar tomorrow. If $K = 100$, $r = 5\%$, and $T = 1$, then $Ke^{-rT} = 95.12$. The "discount" of \$4.88 is the interest you would earn by investing \$95.12 for one year at 5%.

### Verifying Put-Call Parity with Black-Scholes

Now comes the satisfying part: **verification**. We have the analytical Black-Scholes-Merton formulas for European call and put prices. If these formulas are correct, then put-call parity must hold exactly (since BSM is itself derived from no-arbitrage).

Our plan:
1. Implement BSM call and put pricing from scratch.
2. Compute $C - P$ (left side of parity) and $S - Ke^{-rT}$ (right side of parity).
3. Check that their difference is zero (up to floating-point rounding).
4. Repeat across a range of stock prices, strikes, and volatilities.

Recall the Black-Scholes formulas:

$$C = S\,\Phi(d_1) - Ke^{-rT}\,\Phi(d_2)$$
$$P = Ke^{-rT}\,\Phi(-d_2) - S\,\Phi(-d_1)$$

where $\Phi$ is the standard normal CDF, and:

$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

Let us note something beautiful: if you compute $C - P$ from these formulas, the result simplifies *algebraically* to $S - Ke^{-rT}$, using the identity $\Phi(d) + \Phi(-d) = 1$. So the parity is baked into the BSM structure.

> **What to expect:** The difference should be exactly zero (up to floating-point precision, roughly $10^{-14}$). If it is not, there is a bug in our formulas.

In [ ]:
# BSM pricing functions
def bsm_d1_d2(S, K, r, T, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2

def bsm_call(S, K, r, T, sigma):
    d1, d2 = bsm_d1_d2(S, K, r, T, sigma)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

def bsm_put(S, K, r, T, sigma):
    d1, d2 = bsm_d1_d2(S, K, r, T, sigma)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)


def verify_put_call_parity(S, K, r, T, sigma):
    """Verify put-call parity: C - P = S - K*exp(-rT)."""
    C = bsm_call(S, K, r, T, sigma)
    P = bsm_put(S, K, r, T, sigma)
    
    lhs = C - P
    rhs = S - K * np.exp(-r * T)
    error = abs(lhs - rhs)
    
    return C, P, lhs, rhs, error


# Verify across parameter ranges
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2

C, P, lhs, rhs, err = verify_put_call_parity(S0, K, r, T, sigma)
print(f"S={S0}, K={K}, r={r}, T={T}, \u03c3={sigma}")
print(f"  Call = {C:.6f}, Put = {P:.6f}")
print(f"  C - P = {lhs:.6f}")
print(f"  S - K*exp(-rT) = {rhs:.6f}")
print(f"  Error = {err:.2e}")

print(f"\nVerification across different parameters:")
print(f"{'S':>6s} {'K':>6s} {'\u03c3':>6s} {'C':>10s} {'P':>10s} {'C-P':>10s} {'S-Ke^-rT':>10s} {'Error':>10s}")
print("-" * 72)

for S_i in [80, 100, 120]:
    for K_i in [90, 100, 110]:
        for sig_i in [0.15, 0.30]:
            C, P, lhs, rhs, err = verify_put_call_parity(S_i, K_i, r, T, sig_i)
            print(f"{S_i:6.0f} {K_i:6.0f} {sig_i:6.2f} {C:10.4f} {P:10.4f} {lhs:10.4f} {rhs:10.4f} {err:10.2e}")

### Interpreting the Verification Results

The output above confirms that put-call parity holds to machine precision across all 18 parameter combinations tested (3 stock prices x 3 strikes x 2 volatilities). The maximum error is on the order of $10^{-14}$ -- this is purely floating-point rounding, not a real discrepancy.

Some observations from the table:

- **The $C - P$ column depends only on $S$ and $K$, not on $\sigma$.** Look at any two rows with the same $S$ and $K$ but different volatilities -- the $C - P$ values are identical. This confirms that parity is model-free: volatility cancels out.
- **When $S > Ke^{-rT}$, the call is more expensive than the put.** This makes sense: if the stock is above the present value of the strike, the call is more likely to finish in-the-money.
- **When $S < Ke^{-rT}$, the put is more expensive than the call.** The stock is below the discounted strike, favoring the put holder.

> **Key Concept:** This is a powerful sanity check. Any option pricing model that does not satisfy put-call parity has a bug. If you ever implement a new pricing model, verify parity *first* before trusting any other outputs.

> **CFA Exam Tip:** You will never need to verify parity numerically on an exam, but you *will* need to use the formula to find a missing price. If given $C$, $S$, $K$, $r$, $T$, you can find $P$ (and vice versa) without knowing $\sigma$ at all.> **CFA Exam Tip:** Put-call parity is a model-free result — it holds regardless of which option pricing model you use (BSM, binomial, or any other). If your model violates put-call parity, the model is wrong. This makes parity an excellent sanity check for any pricing implementation.

### What Parity Tells Us About Option Relationships

Rearranging the parity equation reveals deep connections:
- $C = P + S - Ke^{-rT}$: A call is equivalent to a put + stock + borrowing
- $P = C - S + Ke^{-rT}$: A put is equivalent to a call + short stock + lending
- $S = C - P + Ke^{-rT}$: Stock can be replicated from options + a bond

These relationships mean that if you know any THREE of the four prices ($C$, $P$, $S$, $Ke^{-rT}$), you can determine the fourth.


---

## 4. Verification and Visualization

Numbers in a table are convincing, but a picture is worth a thousand numbers. Let us verify put-call parity *visually* by plotting both sides of the equation as functions of the stock price.

### What Are We Plotting?

We will create two panels:

**Left panel:** The individual call and put prices ($C$ and $P$) as functions of the stock price $S$. This shows the familiar option price curves:
- The call price increases with $S$ (more valuable when the stock is high).
- The put price decreases with $S$ (more valuable when the stock is low).
- They cross near the at-the-money point ($S \approx K$).

**Right panel:** Both sides of the parity equation plotted together:
- $C - P$ (solid line) -- the left side.
- $S - Ke^{-rT}$ (dashed line) -- the right side.

If parity holds, these two lines should overlap *perfectly*, appearing as a single line.

> **Visual check:** If the solid and dashed lines are indistinguishable, parity is confirmed. Any gap between them would indicate a violation.

In [ ]:
S_range = np.linspace(50, 150, 200)

calls = bsm_call(S_range, K, r, T, sigma)
puts = bsm_put(S_range, K, r, T, sigma)

lhs = calls - puts
rhs = S_range - K * np.exp(-r * T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: C and P
axes[0].plot(S_range, calls, color=PRIMARY, linewidth=2, label='Call C')
axes[0].plot(S_range, puts, color=SECONDARY, linewidth=2, label='Put P')
axes[0].axvline(K, color='gray', linestyle='--', alpha=0.5, label=f'K={K}')
axes[0].set_xlabel('Stock Price S')
axes[0].set_ylabel('Option Price')
axes[0].set_title('BSM Call and Put Prices')
axes[0].legend()

# Right: Parity verification
axes[1].plot(S_range, lhs, color=PRIMARY, linewidth=3, label='C - P')
axes[1].plot(S_range, rhs, color=SECONDARY, linewidth=2, linestyle='--', label=r'$S - Ke^{-rT}$')
axes[1].set_xlabel('Stock Price S')
axes[1].set_ylabel('Value')
axes[1].set_title('Put-Call Parity: Both Sides Overlap Perfectly')
axes[1].legend()

plt.tight_layout()
plt.show()

### Reading the Plots

**Left panel observations:**
- The call price curve is convex and increasing. Deep out-of-the-money calls ($S \ll K$) are nearly worthless; deep in-the-money calls ($S \gg K$) approach $S - Ke^{-rT}$ (the forward value).
- The put price curve is convex and decreasing. Deep in-the-money puts ($S \ll K$) approach $Ke^{-rT} - S$; deep out-of-the-money puts ($S \gg K$) are nearly worthless.
- The vertical dashed line at $K = 100$ marks the at-the-money point. Notice the call and put are NOT equal at $S = K$; they differ by $S - Ke^{-rT} = K(1 - e^{-rT}) \approx 4.88$ (the interest on $K$).

**Right panel observations:**
- The solid line ($C - P$) and the dashed line ($S - Ke^{-rT}$) are *perfectly superimposed*. You cannot tell them apart. This is put-call parity in action.
- The relationship is linear in $S$ on the right side, and the left side (which involves two nonlinear option prices) produces the same straight line. The nonlinearities cancel perfectly.
- When $S = Ke^{-rT} \approx 95.12$, both sides equal zero. This is where $C = P$.

> **Common Mistake:** Students sometimes think $C = P$ when $S = K$ (at-the-money). This is wrong! $C = P$ when $S = Ke^{-rT}$, which is slightly below $K$ for positive interest rates. The difference is the "forward discount."> **Key Concept:** The right panel is the visual proof of put-call parity. The two lines lying perfectly on top of each other means $C - P$ and $S - Ke^{-rT}$ are identical at every stock price. If they ever diverged, there would be an arbitrage opportunity.

> **CFA Exam Tip:** The parity relationship $C - P = S - Ke^{-rT}$ is linear in $S$. This means the spread between call and put prices changes dollar-for-dollar with the stock price. When $S$ increases by \$1, $C - P$ increases by exactly \$1.


---

## 5. Arbitrage from Violations

Now we arrive at the practical payoff of understanding parity: **what do you do when it breaks?**

If put-call parity is violated in the market, there is a **riskless profit** to be made. This is the definition of arbitrage: you make money with zero risk and zero net investment. In efficient markets, such opportunities are fleeting (milliseconds), but understanding them is essential for:
- Market makers who need to maintain consistent quotes.
- Quant traders who build automated arbitrage detection systems.
- Risk managers who use parity as a sanity check on pricing models.

### The Two Cases of Parity Violation

There are exactly two ways parity can break:

---

### Case 1: $C - P > S - Ke^{-rT}$ (the call is "too expensive" relative to the put)

**Strategy:** Sell the overpriced side, buy the underpriced side.

Concretely: **Sell call, buy put, buy stock, borrow $Ke^{-rT}$.**

Let us trace through every cash flow in detail:

| Action | Cash at $t=0$ | Cash at $t=T$ if $S_T > K$ | Cash at $t=T$ if $S_T \leq K$ |
|--------|:------------:|:------------------------:|:-----------------------------:|
| Sell call | $+C$ | $-(S_T - K)$ | $0$ |
| Buy put | $-P$ | $0$ | $+(K - S_T)$ |
| Buy stock | $-S_0$ | $+S_T$ | $+S_T$ |
| Borrow $Ke^{-rT}$ | $+Ke^{-rT}$ | $-K$ | $-K$ |
| **TOTAL** | $C - P - S_0 + Ke^{-rT}$ | $\mathbf{0}$ | $\mathbf{0}$ |

**The initial cash flow $C - P - S_0 + Ke^{-rT} > 0$ is the arbitrage profit.** You pocket this cash today, and at expiry your obligations exactly cancel out -- you owe nothing in either scenario.

This is *literally* free money. You receive cash today and have zero risk in the future.

---

### Case 2: $C - P < S - Ke^{-rT}$ (the put is "too expensive" relative to the call)

**Strategy:** Reverse everything: **Buy call, sell put, short stock, lend $Ke^{-rT}$.**

| Action | Cash at $t=0$ | Cash at $t=T$ if $S_T > K$ | Cash at $t=T$ if $S_T \leq K$ |
|--------|:------------:|:------------------------:|:-----------------------------:|
| Buy call | $-C$ | $+(S_T - K)$ | $0$ |
| Sell put | $+P$ | $0$ | $-(K - S_T)$ |
| Short stock | $+S_0$ | $-S_T$ | $-S_T$ |
| Lend $Ke^{-rT}$ | $-Ke^{-rT}$ | $+K$ | $+K$ |
| **TOTAL** | $-C + P + S_0 - Ke^{-rT}$ | $\mathbf{0}$ | $\mathbf{0}$ |

The initial cash flow $-C + P + S_0 - Ke^{-rT} = S_0 - Ke^{-rT} - (C - P) > 0$ is the arbitrage profit.

> **Key Concept:** The rule is always the same: **sell the expensive side, buy the cheap side.** If $C - P$ is too high, sell the call and buy the put. If $C - P$ is too low, buy the call and sell the put. The stock and bond positions hedge away all risk.

### Why Arbitrage Keeps Markets Honest

In practice, violations of put-call parity are quickly eliminated by arbitrageurs. When a violation appears:
1. Traders detect it (often algorithmically, in microseconds).
2. They execute the arbitrage strategy, which involves buying the cheap instrument and selling the expensive one.
3. This buying/selling pressure pushes prices back toward parity.

The result: in liquid markets like S&P 500 options, parity violations are typically less than the bid-ask spread, making them unprofitable after transaction costs.

> **Common Mistake:** In textbook problems, students sometimes construct the arbitrage portfolio correctly but forget to check the sign of the profit. Always verify: is $C - P - S + Ke^{-rT}$ positive (Case 1) or negative (Case 2)? The sign tells you which direction to trade.

### Constructing the Arbitrage: A Concrete Example

Let us move from theory to practice. Suppose we observe the following market prices:

- Stock $S_0 = 100$
- Strike $K = 100$, Risk-free rate $r = 5\%$, Time $T = 1$ year
- Fair call price (from BSM): approximately \$10.45
- Fair put price (from BSM): approximately \$5.57

Now imagine two scenarios where the market misprices one of the options:

**Scenario A:** The call is overpriced by \$2 (market call = \$12.45). The put is fairly priced.
- Parity says: $C - P$ should equal $S - Ke^{-rT} = 100 - 95.12 = 4.88$.
- But the market has: $C - P = 12.45 - 5.57 = 6.88$, which is too high by \$2.
- Strategy: Sell call, buy put, buy stock, borrow \$95.12. Pocket \$2 immediately.

**Scenario B:** The put is overpriced by \$1.50 (market put = \$7.07). The call is fairly priced.
- Market has: $C - P = 10.45 - 7.07 = 3.38$, which is too low by \$1.50.
- Strategy: Buy call, sell put, short stock, lend \$95.12. Pocket \$1.50 immediately.

The next code cell implements an arbitrage checker that detects violations and reports the exact strategy and cash flows.

> **Key Concept:** This is the essence of **no-arbitrage pricing**. If a riskless profit exists, traders will exploit it until prices adjust and parity is restored. This self-correcting mechanism is what makes parity hold in practice.

In [ ]:
def check_arbitrage(C_market, P_market, S0, K, r, T):
    """Check for put-call parity violations and compute arbitrage.
    
    Returns strategy description and profit.
    """
    pv_K = K * np.exp(-r * T)
    lhs = C_market - P_market
    rhs = S0 - pv_K
    diff = lhs - rhs  # should be 0
    
    if abs(diff) < ATOL:
        return "No arbitrage (parity holds)", 0.0, {}
    
    if diff > 0:
        # Call overpriced: sell call, buy put, buy stock, borrow
        profit = diff
        strategy = "Sell call, buy put, buy stock, borrow K*exp(-rT)"
        flows = {
            'Sell call': C_market,
            'Buy put': -P_market,
            'Buy stock': -S0,
            'Borrow': pv_K,
            'NET PROFIT': profit
        }
    else:
        # Put overpriced: buy call, sell put, short stock, lend
        profit = -diff
        strategy = "Buy call, sell put, short stock, lend K*exp(-rT)"
        flows = {
            'Buy call': -C_market,
            'Sell put': P_market,
            'Short stock': S0,
            'Lend': -pv_K,
            'NET PROFIT': profit
        }
    
    return strategy, profit, flows


# Example: mispriced call
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2
C_fair = bsm_call(S0, K, r, T, sigma)
P_fair = bsm_put(S0, K, r, T, sigma)

print("=" * 60)
print("Case 1: Call overpriced by $2")
print("=" * 60)
C_over = C_fair + 2.0
strategy, profit, flows = check_arbitrage(C_over, P_fair, S0, K, r, T)
print(f"Strategy: {strategy}")
print(f"\nCash flows at t=0:")
for action, amount in flows.items():
    print(f"  {action:30s}: {amount:+.4f}")

print(f"\n{'=' * 60}")
print("Case 2: Put overpriced by $1.5")
print("=" * 60)
P_over = P_fair + 1.5
strategy, profit, flows = check_arbitrage(C_fair, P_over, S0, K, r, T)
print(f"Strategy: {strategy}")
print(f"\nCash flows at t=0:")
for action, amount in flows.items():
    print(f"  {action:30s}: {amount:+.4f}")

### Interpreting the Arbitrage Output

The output above shows two concrete arbitrage examples:

**Case 1 (call overpriced by \$2):** The strategy is to sell the expensive call, buy the cheap put, buy the stock, and borrow $Ke^{-rT}$. The net cash flow at $t=0$ is exactly +\$2.00 -- you pocket this immediately. At expiry, your obligations cancel to zero in every scenario.

**Case 2 (put overpriced by \$1.50):** The strategy reverses: buy the call, sell the expensive put, short the stock, and lend $Ke^{-rT}$. Net profit: \$1.50.

Notice how each cash flow line has a $+$ or $-$ sign. Positive means you receive cash; negative means you pay cash. The "NET PROFIT" line is always positive -- that is the hallmark of arbitrage.

> **CFA Exam Tip:** On the CFA exam, arbitrage questions often ask you to state the strategy in words: "Sell the overpriced option, buy the underpriced option, and hedge with stock/bonds." Practice identifying which side is overpriced by checking if $C - P \gtrless S - Ke^{-rT}$.> **Key Concept:** In practice, arbitrage opportunities like these are rare and fleeting. High-frequency traders and market makers monitor put-call parity continuously. When a violation appears, they exploit it within milliseconds, pushing prices back into alignment. Transaction costs (bid-ask spreads, commissions, borrowing costs) mean the violation must exceed a minimum threshold before arbitrage is profitable.

> **CFA Exam Tip:** On the exam, you may be asked to construct the arbitrage portfolio step by step. Remember the rule: **buy the cheap side, sell the expensive side.** If $C - P > S - Ke^{-rT}$, the left side is expensive — sell the call, buy the put, buy the stock, borrow. If $C - P < S - Ke^{-rT}$, the left side is cheap — buy the call, sell the put, short the stock, lend.


### Verifying Arbitrage Bounds and the Zero-Payoff Portfolio

Even without put-call parity, we can establish **lower bounds** on option prices using no-arbitrage arguments.

For a European call: $C \geq \max(0, S - Ke^{-rT})$

For a European put: $P \geq \max(0, Ke^{-rT} - S)$

> **Why?** If a call were priced below $S - Ke^{-rT}$, you could buy the call, short the stock, and invest $Ke^{-rT}$ at the risk-free rate. At expiry, you are guaranteed a profit. Since arbitrage is impossible in equilibrium, the bound must hold.

### The Arbitrage Portfolio at Expiry

Before looking at bounds visually, let us first confirm that the arbitrage portfolio from Case 1 truly has zero net payoff at expiry. This is the "proof by picture" that the arbitrage is riskless.

We plot the four legs of the portfolio (short call, long put, long stock, repay bond) individually as dashed lines, plus their sum as a thick black line. If parity-based arbitrage works, the thick black line should be flat at zero.

> **What to look for:** Each individual leg has a kinked or sloped payoff, but they combine to produce a perfectly flat line at zero. This is the magic of the replicating portfolio -- the risks cancel *exactly*.

### Cash Flow Explanation in Detail

The cash flow table in the output shows that:
- **Today:** You receive a net positive cash flow (free money!)
- **At expiry:** Your payoff is exactly zero regardless of whether $S_T > K$ or $S_T \leq K$

This is a pure arbitrage: riskless profit with zero investment. In practice, transaction costs, bid-ask spreads, and execution risk mean you need a sufficiently large violation to profit. But the principle ensures that parity holds approximately in real markets.

**Real-world frictions that limit arbitrage:**
- **Bid-ask spreads:** You buy at the ask and sell at the bid, losing the spread on each leg.
- **Transaction costs:** Commissions, exchange fees, clearing fees.
- **Borrowing costs:** Shorting stock requires borrowing shares, which has a cost (the "borrow rate").
- **Execution risk:** By the time you execute all four legs, prices may have moved.
- **Margin requirements:** Brokers require collateral for short positions.

Despite these frictions, parity violations beyond a few cents are rare in liquid markets. The existence of arbitrageurs *enforces* the relationship.

> **Common Mistake:** Students sometimes forget to include the time value of money. The comparison is $C - P = S - Ke^{-rT}$, NOT $C - P = S - K$. The discount factor matters! For a 1-year option with $K = 100$ and $r = 5\%$, the difference between $K$ and $Ke^{-rT}$ is about \$4.88 -- far from negligible.> **Common Mistake:** When constructing the arbitrage, students often forget the time value of money. The comparison is $C - P$ vs $S - Ke^{-rT}$, NOT $S - K$. The present value of the strike matters because you don't pay $K$ until expiry.

> **Important:** The arbitrage is truly riskless — the net payoff at expiry is zero in ALL scenarios. The profit is locked in today at inception. This is what makes it different from speculation, which involves taking a view on future prices.


In [ ]:
# Verify zero net payoff at expiry for the arbitrage portfolio
S_T = np.linspace(50, 150, 200)

# Case 1 portfolio: short call, long put, long stock, short bond
call_payoff = -np.maximum(S_T - K, 0)  # short call
put_payoff = np.maximum(K - S_T, 0)     # long put
stock_payoff = S_T                       # long stock
bond_payoff = -K * np.ones_like(S_T)     # repay loan

total = call_payoff + put_payoff + stock_payoff + bond_payoff

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(S_T, call_payoff, '--', color=PRIMARY, label='Short Call')
ax.plot(S_T, put_payoff, '--', color=SECONDARY, label='Long Put')
ax.plot(S_T, stock_payoff, '--', color=TERTIARY, label='Long Stock')
ax.plot(S_T, bond_payoff, '--', color=ACCENT, label='Repay Bond')
ax.plot(S_T, total, 'k-', linewidth=3, label=f'Net Payoff = {total[0]:.0f}')
ax.axhline(0, color='gray', alpha=0.3)
ax.set_xlabel('Stock Price at Expiry $S_T$')
ax.set_ylabel('Payoff')
ax.set_title('Arbitrage Portfolio Payoffs at Expiry (zero net = riskless profit at t=0)')
ax.legend()
plt.tight_layout()
plt.show()

### Reading the Arbitrage Payoff Diagram

The plot above is one of the most important diagrams in options theory. Here is what each line represents:

- **Short Call (blue dashed):** You sold a call, so you *lose* money when $S_T > K$. The payoff is $-\max(S_T - K, 0)$, which is zero for $S_T \leq K$ and slopes downward for $S_T > K$.

- **Long Put (coral dashed):** You bought a put, so you *gain* money when $S_T < K$. The payoff is $\max(K - S_T, 0)$, which slopes downward from $K$ to zero as $S_T$ goes from 0 to $K$, then stays at zero.

- **Long Stock (green dashed):** You own the stock. Payoff is simply $S_T$ -- a straight line with slope 1.

- **Repay Bond (gold dashed):** You borrowed $Ke^{-rT}$ today, so at expiry you repay $K$. This is a flat line at $-K$.

- **Net Payoff (black solid):** The sum of all four legs. It is identically zero for every value of $S_T$. This confirms that the portfolio has no risk at expiry -- all the profit comes from the initial cash flow at $t=0$.

> **Key Concept:** The flat black line at zero is the visual proof that this arbitrage is truly riskless. No matter where the stock ends up -- whether it crashes to \$50 or soars to \$150 -- your net obligation at expiry is exactly zero. The arbitrage profit was locked in at time zero.> **Common Mistake:** Don't confuse "zero net payoff at expiry" with "zero profit." The profit was already captured today at inception (the positive cash flow when setting up the portfolio). The zero payoff at expiry simply means there is no residual risk — the position is perfectly hedged.


---

## 6. Lower Bounds on Option Prices

Put-call parity is an *equality* -- it pins down the relationship between calls and puts exactly. But even without knowing the put price, we can establish strict **lower bounds** on the call price (and vice versa) using no-arbitrage arguments alone.

### The Bounds

For European options on a non-dividend-paying stock:

$$C \geq \max\!\left(0,\; S_0 - K e^{-rT}\right)$$

$$P \geq \max\!\left(0,\; K e^{-rT} - S_0\right)$$

### Why $\max(0, \ldots)$?

Option prices can never be negative (you would never pay someone to take a right from you). So the bound is the maximum of zero and the "intrinsic-like" value.

### Proof for the Call Lower Bound

Suppose (for contradiction) that $C < S_0 - Ke^{-rT}$.

Consider this portfolio: **buy the call, short the stock, invest $Ke^{-rT}$ at the risk-free rate.**

- Cost today: $C - S_0 + Ke^{-rT} < 0$ (you receive money because $C < S_0 - Ke^{-rT}$).
- At expiry: $\max(S_T - K, 0) - S_T + K = \max(0, K - S_T) \geq 0$.

So you receive cash today AND have a non-negative payoff at expiry. This is an arbitrage -- a contradiction. Therefore, $C \geq S_0 - Ke^{-rT}$.

Combined with $C \geq 0$ (options cannot have negative prices), we get:
$$C \geq \max(0,\, S_0 - Ke^{-rT})$$

### Important Distinction: Lower Bound vs. Intrinsic Value

The **intrinsic value** of a call is $\max(0, S - K)$, while the **lower bound** is $\max(0, S - Ke^{-rT})$. Notice the lower bound is *higher* than the intrinsic value (because $Ke^{-rT} < K$). This gap represents the "time value of money" component.

For puts, the intrinsic value $\max(0, K - S)$ can actually exceed the European put price (this is why American puts can be worth more than European puts -- early exercise lets you capture the full intrinsic value).

> **Key Concept:** The lower bound tells you the absolute minimum a fairly-priced option can cost. If you ever see an option trading below its lower bound, there is a guaranteed arbitrage opportunity.

> **CFA Exam Tip:** The call lower bound $\max(0, S - Ke^{-rT})$ is NOT the same as the intrinsic value $\max(0, S - K)$. The lower bound uses the *discounted* strike. This distinction trips up many candidates.### Deriving the Call Lower Bound

Why must $C \geq S - Ke^{-rT}$? Suppose it weren't — suppose $C < S - Ke^{-rT}$. Then you could:

1. Buy the call for $C$
2. Short the stock, receiving $S$
3. Invest $Ke^{-rT}$ at the risk-free rate (grows to $K$ at expiry)

**Net cash today:** $S - C - Ke^{-rT} > 0$ (positive by assumption)

**At expiry:**
- If $S_T > K$: Exercise call, pay $K$, get stock, return to short. Net = 0
- If $S_T \leq K$: Let call expire, buy stock at $S_T$, return to short, collect $K$ from investment. Net = $K - S_T \geq 0$

You started with positive cash and end with non-negative cash — **riskless profit**. Since this can't exist in equilibrium, the bound must hold.

> **Key Concept:** The lower bounds are weaker than put-call parity (they're inequalities, not equalities). But they hold for BOTH American and European options, making them more broadly applicable.


### American Options: Why Parity Breaks Down

Everything we have discussed so far applies to **European** options, which can only be exercised at expiry. What happens with **American** options, which can be exercised at any time?

The short answer: put-call parity becomes an **inequality** rather than an equality.

### Why Parity Fails for American Options

The replicating portfolio argument relied on comparing payoffs *at expiry*. For European options, that is the only relevant time. But American options can be exercised early, which creates an asymmetry:

- **American call on a non-dividend-paying stock:** Early exercise is *never* optimal. Why? Because the call is always worth at least $S - Ke^{-rT} > S - K$ (its lower bound exceeds its intrinsic value). You are better off selling the call than exercising it. So $C_A = C_E$ -- the American call equals the European call.

- **American put:** Early exercise *can* be optimal. Consider a stock that has dropped to nearly zero. The put is worth nearly $K$ if exercised now. But the European put can only deliver $K$ at expiry, which has a present value of $Ke^{-r(T-t)} < K$. By exercising early, you get $K$ in cash *today* and can invest it at the risk-free rate. The **early exercise premium** makes $P_A > P_E$.

### The American Put-Call Parity Bounds

Since $C_A = C_E$ but $P_A \geq P_E$, we get:

$$S_0 - K \leq C_A - P_A \leq S_0 - Ke^{-rT}$$

The upper bound comes from: $C_A - P_A \leq C_E - P_E = S_0 - Ke^{-rT}$ (since $P_A \geq P_E$ makes $C_A - P_A$ smaller).

The lower bound comes from a separate no-arbitrage argument: if $C_A - P_A < S_0 - K$, one could construct an arbitrage by buying the call, selling the put, shorting the stock, and investing $K$ at the risk-free rate.

### When Is Early Exercise of a Put Optimal?

Roughly speaking, early exercise of a put is optimal when the stock price is sufficiently low. The critical stock price (below which you should exercise) decreases as $T$ increases and increases as $r$ increases. The intuition:

- **Higher $r$:** The benefit of receiving $K$ early (and investing at rate $r$) is greater.
- **Lower $S$:** The put is deep in-the-money, so the time value of waiting is small relative to the interest on $K$.

> **CFA Exam Tip:** For American options on non-dividend-paying stocks, put-call parity becomes a set of bounds, not an equation. The early exercise premium on the put creates the gap. Remember: American call = European call (no early exercise), but American put > European put (early exercise has value).

> **Common Mistake:** Some students think American options always satisfy $C_A - P_A = S - Ke^{-rT}$. They do not! The correct statement is that this is only the *upper bound*, with $S - K$ as the lower bound.

### Visualizing Bounds and American Options

The next plot shows call and put prices from Black-Scholes, overlaid with their lower bounds and the feasible pricing region. The shaded area represents all prices that are consistent with no-arbitrage. The BSM curve lies within this region, above the lower bound and below the upper bound.> **Key Concept:** The key difference is that an American put may be optimally exercised early. If the stock price drops very low, the put holder can exercise, receive $K$ in cash, and invest it at the risk-free rate. The time value of that cash can exceed the remaining optionality of holding the put. This creates an **early exercise premium** that doesn't exist for European puts.

For American calls on non-dividend-paying stocks, early exercise is NEVER optimal. Why? Because:
1. Exercising destroys the remaining time value
2. You pay $K$ today instead of $Ke^{-rT}$ later (losing the interest on $K$)
3. You give up the downside protection (the call can't lose more than its premium; exercised stock can)

> **Common Mistake:** Students often think "if the stock is way above the strike, you should exercise the call early to lock in profits." This is wrong for non-dividend-paying stocks — you're always better off selling the call in the market, which captures both intrinsic and time value.


In [ ]:
S_range = np.linspace(50, 150, 300)
K, r, T, sigma = 100, 0.05, 1.0, 0.2
pv_K = K * np.exp(-r * T)

C = bsm_call(S_range, K, r, T, sigma)
P = bsm_put(S_range, K, r, T, sigma)

# Lower bounds
call_lb = np.maximum(0, S_range - pv_K)
put_lb = np.maximum(0, pv_K - S_range)

# Upper bounds
call_ub = S_range  # call can't be worth more than the stock
put_ub = pv_K * np.ones_like(S_range)  # put can't be worth more than PV(K)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Call bounds
axes[0].fill_between(S_range, call_lb, call_ub, alpha=0.1, color=PRIMARY, label='Feasible region')
axes[0].plot(S_range, C, color=PRIMARY, linewidth=2, label='BSM Call')
axes[0].plot(S_range, call_lb, '--', color=SECONDARY, linewidth=1.5, label=r'Lower: $\max(0, S-Ke^{-rT})$')
axes[0].plot(S_range, np.maximum(S_range - K, 0), ':', color='gray', linewidth=1, label='Intrinsic')
axes[0].set_xlabel('Stock Price S')
axes[0].set_ylabel('Price')
axes[0].set_title('Call Option Bounds')
axes[0].legend(fontsize=9)
axes[0].set_ylim(-5, 60)

# Put bounds
axes[1].fill_between(S_range, put_lb, put_ub, alpha=0.1, color=SECONDARY, label='Feasible region')
axes[1].plot(S_range, P, color=SECONDARY, linewidth=2, label='BSM Put')
axes[1].plot(S_range, put_lb, '--', color=PRIMARY, linewidth=1.5, label=r'Lower: $\max(0, Ke^{-rT}-S)$')
axes[1].plot(S_range, np.maximum(K - S_range, 0), ':', color='gray', linewidth=1, label='Intrinsic')
axes[1].set_xlabel('Stock Price S')
axes[1].set_ylabel('Price')
axes[1].set_title('Put Option Bounds')
axes[1].legend(fontsize=9)
axes[1].set_ylim(-5, 60)

plt.tight_layout()
plt.show()

### Reading the Bounds Plots

**Left panel (Call Bounds):**
- The light blue shaded region is the "feasible" zone: any call price in this region is consistent with no-arbitrage.
- The solid blue line (BSM call) lies within this region, hugging the lower bound for deep in-the-money options and curving away for out-of-the-money options.
- The dashed coral line is the lower bound $\max(0, S - Ke^{-rT})$. Notice it is *above* the gray dotted intrinsic value line $\max(0, S - K)$ -- the gap between them is $K - Ke^{-rT} = K(1 - e^{-rT})$, which is the interest earned on $K$.
- The upper bound for a call is the stock price itself (a call can never be worth more than the underlying stock).

**Right panel (Put Bounds):**
- The shaded coral region shows the feasible zone for put prices.
- Note that the BSM put curve dips *below* the intrinsic value line (gray dotted) for deep in-the-money puts! This is because a European put forces you to wait until expiry, and the time value of money erodes the benefit of receiving $K$. This is exactly why American puts can be worth more -- you can exercise early to capture the full intrinsic value.
- The upper bound for a European put is $Ke^{-rT}$ (the present value of the maximum payoff $K$).

> **Key Concept:** For calls, the no-arbitrage lower bound always exceeds the intrinsic value. For puts, the European price can actually fall below intrinsic value for deep in-the-money options. This asymmetry is why early exercise matters for puts but not calls (on non-dividend-paying stocks).

---

## 7. American Options: Numerical Verification

We have established that American put-call parity is an inequality:

$$S_0 - K \leq C_A - P_A \leq S_0 - K e^{-rT}$$

Let us now verify this numerically using a **binomial tree** to price American options. The CRR (Cox-Ross-Rubinstein) binomial model is the standard approach:

1. Build a recombining tree of stock prices with $N$ time steps.
2. At the terminal nodes, compute the option's intrinsic value.
3. Working backwards through the tree, at each node compute the continuation value (discounted expected value under the risk-neutral measure) and the intrinsic value.
4. The American option value at each node is the maximum of continuation and intrinsic (exercise if intrinsic > continuation).

### Key Facts to Verify

- **American call = European call** (for non-dividend-paying stocks). Early exercise of a call is never optimal because the call's value always exceeds its intrinsic value. The early exercise premium for the call should be approximately zero.

- **American put > European put.** The early exercise premium is positive, especially for deep in-the-money puts. This is the premium you pay for the *option* to exercise early.

- **$C_A - P_A$ falls between the bounds.** The exact value depends on how large the early exercise premium is.

> **Key Concept:** The binomial tree with $N = 500$ steps provides high accuracy (convergence to the continuous-time limit). For American options, there is no closed-form solution, so numerical methods like trees or finite differences are essential.

### Adjusting for Dividends

So far, we have assumed the stock pays no dividends. But most stocks do pay dividends, and this significantly affects put-call parity.

### Why Dividends Matter

When a stock pays a dividend, the stock price drops by approximately the dividend amount on the ex-dividend date. This affects option values asymmetrically:

- **Call holders lose:** They do not receive the dividend (only stockholders do), and the post-dividend stock price drop hurts their call.
- **Put holders gain:** The lower post-dividend stock price increases their put's value.

### Modified Parity with Known Discrete Dividends

If the stock pays known dividends $D_i$ at times $t_i < T$, put-call parity becomes:

$$C - P = (S_0 - D_{\text{PV}}) - Ke^{-rT}$$

where $D_{\text{PV}} = \sum_i D_i e^{-r t_i}$ is the present value of all dividends paid during the option's life.

Equivalently, replace $S_0$ with $S_0 - D_{\text{PV}}$ in all formulas -- the "adjusted spot price."

### Worked Example with Dividends

Suppose $S_0 = 100$, $K = 100$, $r = 5\%$, $T = 1$ year, and the stock pays \$1.50 quarterly (at $t = 0.25, 0.50, 0.75$).

**Step 1: Compute PV of dividends.**

$$D_{\text{PV}} = 1.50 \cdot e^{-0.05 \times 0.25} + 1.50 \cdot e^{-0.05 \times 0.50} + 1.50 \cdot e^{-0.05 \times 0.75}$$
$$= 1.50 \times 0.9876 + 1.50 \times 0.9753 + 1.50 \times 0.9632 = 1.4814 + 1.4629 + 1.4448 = 4.3891$$

**Step 2: Adjusted spot price.**

$$S_{\text{adj}} = S_0 - D_{\text{PV}} = 100 - 4.3891 = 95.6109$$

**Step 3: Apply parity with adjusted spot.**

$$C - P = S_{\text{adj}} - Ke^{-rT} = 95.6109 - 95.1229 = 0.4880$$

Compare to the no-dividend case: $C - P = 100 - 95.1229 = 4.8771$. The dividends shift the relationship by about \$4.39 (the PV of dividends).

> **Key Concept:** Dividends reduce the call value and increase the put value. The intuition: dividends are cash flows that the stockholder receives but the call holder does not. The call holder is "missing out" on dividend income.

> **CFA Exam Tip:** When dividends are present, always use the *adjusted* spot price $S_0 - D_{\text{PV}}$ in the parity formula. A common exam trap is to use the unadjusted spot price.

The next two code cells verify these results numerically: first the American option bounds, then the dividend adjustment.> **Important:** The dividend adjustment is critical in practice because most stocks pay dividends. Ignoring dividends when checking parity will show spurious "violations" that aren't real arbitrage opportunities.

**Types of dividend adjustments:**

| Dividend type | Parity adjustment | When used |
|:---|:---|:---|
| Known discrete dividends | $S_0 - PV(\text{divs})$ replaces $S_0$ | Individual stock options |
| Continuous dividend yield $q$ | $S_0 e^{-qT}$ replaces $S_0$ | Index options, FX options |

> **CFA Exam Tip:** For options on stock indices, the continuous yield version is standard: $C - P = S_0 e^{-qT} - Ke^{-rT}$. The dividend yield $q$ reduces the effective stock price for the call holder (who doesn't receive dividends).


In [ ]:
def binomial_american(S0, K, r, T, sigma, N, option_type='put'):
    """Price an American option via CRR binomial tree."""
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    q = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    
    j = np.arange(N + 1)
    ST = S0 * u**j * d**(N - j)
    
    if option_type == 'call':
        V = np.maximum(ST - K, 0.0)
    else:
        V = np.maximum(K - ST, 0.0)
    
    for n in range(N - 1, -1, -1):
        j_arr = np.arange(n + 1)
        S_n = S0 * u**j_arr * d**(n - j_arr)
        cont = disc * (q * V[1:n+2] + (1 - q) * V[0:n+1])
        if option_type == 'call':
            intrinsic = np.maximum(S_n - K, 0.0)
        else:
            intrinsic = np.maximum(K - S_n, 0.0)
        V = np.maximum(cont, intrinsic)
    
    return V[0]


# Verify American put-call parity bounds
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2
N = 500

C_A = binomial_american(S0, K, r, T, sigma, N, 'call')
P_A = binomial_american(S0, K, r, T, sigma, N, 'put')
C_E = bsm_call(S0, K, r, T, sigma)
P_E = bsm_put(S0, K, r, T, sigma)

lower = S0 - K
upper = S0 - K * np.exp(-r * T)
diff_A = C_A - P_A

print(f"European:  C_E = {C_E:.4f}, P_E = {P_E:.4f}, C_E - P_E = {C_E - P_E:.4f}")
print(f"American:  C_A = {C_A:.4f}, P_A = {P_A:.4f}, C_A - P_A = {diff_A:.4f}")
print(f"\nEarly exercise premium (put): {P_A - P_E:.4f}")
print(f"Early exercise premium (call): {C_A - C_E:.6f} (\u2248 0)")
print(f"\nAmerican bounds: {lower:.4f} \u2264 {diff_A:.4f} \u2264 {upper:.4f}")
print(f"  Lower bound satisfied: {lower <= diff_A + ATOL}")
print(f"  Upper bound satisfied: {diff_A <= upper + ATOL}")

### Interpreting the American Option Results

The output confirms several important facts:

1. **American call equals European call.** The early exercise premium for the call is essentially zero ($\approx 10^{-4}$ or less, which is binomial tree discretization error). This confirms: *it is never optimal to exercise an American call early on a non-dividend-paying stock.*

2. **American put exceeds European put.** The early exercise premium for the put is positive (typically around \$0.30-\$0.50 for these parameters). This is the value of the *right* to exercise early.

3. **The bounds hold.** $C_A - P_A$ falls strictly between $S_0 - K = 0$ (the lower bound) and $S_0 - Ke^{-rT} \approx 4.88$ (the upper bound). The European value $C_E - P_E = 4.88$ is at the upper bound (as expected, since $P_A > P_E$ pushes $C_A - P_A$ below $C_E - P_E$).

> **Key Concept:** The gap between the upper and lower bounds is $K(1 - e^{-rT}) \approx 4.88$. For short-dated options (small $T$) or low interest rates (small $r$), this gap shrinks, and American parity approaches European parity.

### Synthetic Positions from Parity

Put-call parity is not just a theoretical curiosity -- it is a **practical tool** used daily by traders, market makers, and risk managers. By rearranging the parity equation, you can create a **synthetic** version of any position from the other three instruments.

Think of it this way: parity says $C$, $P$, $S$, and $Ke^{-rT}$ are all connected. If you know any three, you can replicate the fourth. This is like having four puzzle pieces where any three can reconstruct the fourth.

### The Four Synthetic Positions

Starting from $C + Ke^{-rT} = P + S$, we can solve for each variable:

| What You Want | How to Build It | Parity Rearrangement |
|:---|:---|:---|
| **Synthetic long stock** | Long call + short put + lend $Ke^{-rT}$ | $S = C - P + Ke^{-rT}$ |
| **Synthetic long call** | Long stock + long put + borrow $Ke^{-rT}$ | $C = S + P - Ke^{-rT}$ |
| **Synthetic long put** | Short stock + long call + lend $Ke^{-rT}$ | $P = C - S + Ke^{-rT}$ |
| **Synthetic risk-free bond** | Long stock + long put - long call | $Ke^{-rT} = S + P - C$ |

### Why Use Synthetics? The "I Want X but I'll Build It from Y and Z" Principle

You might wonder: why build a synthetic stock when you can just buy the actual stock? Several reasons:

1. **Arbitrage exploitation.** If the synthetic is cheaper than the real thing, buy the synthetic and short the real thing (or vice versa).

2. **Liquidity.** Sometimes options are more liquid than the underlying stock (or vice versa). The synthetic route may have lower transaction costs.

3. **Short-selling restrictions.** In some markets, short-selling stocks is restricted or banned. But you can create a synthetic short position using options (long put + short call + borrow). This was famously used during short-selling bans in 2008.

4. **Tax optimization.** In some jurisdictions, the tax treatment of options differs from stocks. A synthetic stock position may have tax advantages.

5. **Leverage.** Options provide inherent leverage. A synthetic stock position via options requires less capital than buying the actual stock.

> **CFA Exam Tip:** A classic exam question: "How do you create a synthetic long call?" Answer: Buy stock, buy put, borrow $Ke^{-rT}$. The payoff at expiry is $S_T + \max(K - S_T, 0) - K = \max(S_T - K, 0)$, which is exactly a call payoff.

> **Common Mistake:** Confusing the direction of the bond. In a synthetic long stock ($S = C - P + Ke^{-rT}$), the $+Ke^{-rT}$ means you *lend* (buy a bond). In a synthetic long call ($C = S + P - Ke^{-rT}$), the $-Ke^{-rT}$ means you *borrow*. Getting the sign wrong flips the entire strategy.

Let us now visualize American option behavior and then verify synthetic positions graphically.
> **Key Concept:** The early exercise premium on an American put increases as interest rates rise (the opportunity cost of not exercising grows), as the option goes deeper in the money, and as time to expiry increases. For American calls on non-dividend-paying stocks, the early exercise premium is always zero.


In [ ]:
# Visualize American vs European and bounds
S_range = np.linspace(60, 140, 50)
C_Am = np.array([binomial_american(s, K, r, T, sigma, 200, 'call') for s in S_range])
P_Am = np.array([binomial_american(s, K, r, T, sigma, 200, 'put') for s in S_range])
C_Eu = bsm_call(S_range, K, r, T, sigma)
P_Eu = bsm_put(S_range, K, r, T, sigma)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# C_A - P_A with bounds
diff_am = C_Am - P_Am
lb = S_range - K
ub = S_range - K * np.exp(-r * T)

axes[0].fill_between(S_range, lb, ub, alpha=0.15, color=TERTIARY, label='Feasible region')
axes[0].plot(S_range, diff_am, color=PRIMARY, linewidth=2, label=r'$C_A - P_A$')
axes[0].plot(S_range, C_Eu - P_Eu, '--', color=SECONDARY, linewidth=2, label=r'$C_E - P_E$')
axes[0].plot(S_range, lb, ':', color='gray', label=r'$S - K$')
axes[0].plot(S_range, ub, ':', color='black', label=r'$S - Ke^{-rT}$')
axes[0].set_xlabel('S')
axes[0].set_ylabel('C - P')
axes[0].set_title('American Put-Call Parity Bounds')
axes[0].legend(fontsize=9)

# Early exercise premium
axes[1].plot(S_range, P_Am - P_Eu, color=SECONDARY, linewidth=2, label='Put premium $P_A - P_E$')
axes[1].plot(S_range, C_Am - C_Eu, color=PRIMARY, linewidth=2, label='Call premium $C_A - C_E$')
axes[1].set_xlabel('S')
axes[1].set_ylabel('Early Exercise Premium')
axes[1].set_title('American Early Exercise Premium')
axes[1].legend()

plt.tight_layout()
plt.show()

### Reading the American Option Plots

**Left panel (American Parity Bounds):**
- The green shaded region shows the feasible zone for $C_A - P_A$.
- The solid blue line ($C_A - P_A$) lies *below* the dashed coral line ($C_E - P_E$). This is because $P_A > P_E$ while $C_A \approx C_E$, so the American spread is smaller.
- The European spread $C_E - P_E$ coincides with the upper bound $S - Ke^{-rT}$ (as expected from European parity).
- For low stock prices, $C_A - P_A$ approaches the lower bound $S - K$. This makes sense: when the stock is low, the American put's early exercise premium is largest, pushing $P_A$ further above $P_E$.

**Right panel (Early Exercise Premium):**
- The coral line (put premium) is largest for low stock prices (deep in-the-money puts) and decreases toward zero for high stock prices (out-of-the-money puts).
- The blue line (call premium) is essentially zero everywhere -- confirming that early exercise of an American call on a non-dividend-paying stock is never optimal.
- The small wiggles in the blue line are binomial tree discretization artifacts (we used $N = 200$ steps for speed).

> **Common Mistake:** Assuming the American early exercise premium is constant. It is not -- it depends heavily on moneyness. Deep in-the-money puts have the largest premium; out-of-the-money puts have almost no premium.

---

## 8. Dividend Adjustments: Numerical Verification

With known discrete dividends $D_i$ paid at times $t_i < T$, put-call parity becomes:

$$C - P = S_0 - D_{\text{PV}} - K e^{-rT}$$

where $D_{\text{PV}} = \sum_i D_i\, e^{-r t_i}$ is the present value of dividends.

Equivalently, replace $S_0$ with $S_0 - D_{\text{PV}}$ in all formulas.

### Implementation Strategy

The simplest approach: compute the PV of dividends, subtract from $S_0$ to get the adjusted spot, then apply standard BSM formulas to the adjusted spot. This works because dividends reduce the "effective" stock price seen by option holders.

The code below:
1. Computes $D_{\text{PV}}$ from a list of (time, amount) dividend pairs.
2. Prices call and put using BSM on the adjusted spot $S_{\text{adj}} = S_0 - D_{\text{PV}}$.
3. Verifies that parity holds with the adjusted formula.
4. Compares with the no-dividend case to show the impact of dividends on option prices.

In [ ]:
def put_call_parity_with_dividends(S0, K, r, T, sigma, dividends):
    """Verify put-call parity with discrete dividends.
    
    Parameters
    ----------
    dividends : list of (time, amount) tuples
    """
    # Present value of dividends
    D_PV = sum(D * np.exp(-r * t) for t, D in dividends)
    
    # Adjusted spot
    S_adj = S0 - D_PV
    
    C = bsm_call(S_adj, K, r, T, sigma)
    P = bsm_put(S_adj, K, r, T, sigma)
    
    lhs = C - P
    rhs = S0 - D_PV - K * np.exp(-r * T)
    
    return C, P, lhs, rhs, abs(lhs - rhs), D_PV


# Example with quarterly dividends
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2
dividends = [(0.25, 1.5), (0.5, 1.5), (0.75, 1.5)]  # $1.50 quarterly

C, P, lhs, rhs, error, D_PV = put_call_parity_with_dividends(S0, K, r, T, sigma, dividends)

print(f"Dividends: {dividends}")
print(f"PV of dividends: {D_PV:.4f}")
print(f"Adjusted spot: S - D_PV = {S0 - D_PV:.4f}")
print(f"\nCall = {C:.4f}, Put = {P:.4f}")
print(f"C - P = {lhs:.6f}")
print(f"S - D_PV - K*exp(-rT) = {rhs:.6f}")
print(f"Error = {error:.2e}")

# Compare with no-dividend case
C_nd = bsm_call(S0, K, r, T, sigma)
P_nd = bsm_put(S0, K, r, T, sigma)
print(f"\nWithout dividends: C = {C_nd:.4f}, P = {P_nd:.4f}")
print(f"With dividends:    C = {C:.4f}, P = {P:.4f}")
print(f"\nDividends reduce call value and increase put value.")

### Interpreting the Dividend Results

The output confirms several important points:

1. **Parity holds with dividends** -- the error is at machine precision ($\sim 10^{-14}$). This confirms our adjustment is correct.

2. **Dividends reduce the call price.** The call is cheaper because the call holder does not receive dividends during the option's life. The stock price will drop by approximately the dividend amount on each ex-date, reducing the expected payoff of the call.

3. **Dividends increase the put price.** The put benefits from the lower expected stock price at expiry. The stock drops are "in favor" of the put holder.

4. **The PV of dividends** ($\approx$ \$4.39 for three quarterly \$1.50 payments) is subtracted from the spot in the parity formula. This is a significant adjustment -- roughly the same magnitude as the interest rate discount on $K$.

> **Key Concept:** Dividends shift the balance of power from calls to puts. A high-dividend stock will have relatively cheaper calls and more expensive puts compared to a non-dividend stock. This is because dividends represent cash flows that the stockholder gets but the call holder misses.

> **Common Mistake:** Using the *total* dividend amount instead of the *present value* of dividends. Each dividend must be discounted to today at the risk-free rate. For short-dated options this barely matters, but for long-dated LEAPS, the discounting can be significant.
> **CFA Exam Tip:** When a question mentions "ex-dividend date," the stock price drops by approximately the dividend amount on that date. For put-call parity, what matters is the present value of dividends paid DURING the option's life — dividends paid after expiry are irrelevant.


---

## 9. Synthetic Positions: Visual Verification

We now arrive at one of the most practically useful applications of put-call parity: **synthetic positions**.

Put-call parity lets us create **synthetic** versions of any position from the other three:

| Synthetic Position | Construction |
|-------------------|-------------|
| **Synthetic long stock** | Long call + short put + lend $Ke^{-rT}$ |
| **Synthetic short stock** | Short call + long put + borrow $Ke^{-rT}$ |
| **Synthetic long call** | Long put + long stock + borrow $Ke^{-rT}$ |
| **Synthetic long put** | Long call + short stock + lend $Ke^{-rT}$ |

These are used in practice for:
- **Exploiting mispricings** -- if the synthetic is cheaper, buy it and sell the real thing.
- **Creating positions when one instrument is illiquid** -- build it from the other three.
- **Tax or regulatory optimization** -- different tax treatment for options vs. stocks.
- **Circumventing short-selling bans** -- synthetic short stock via options.

### What the Plots Show

The next cell creates four panels, one for each synthetic position. In each panel:
- **Dashed lines** show the individual components of the synthetic construction.
- **Thick black line** shows the synthetic payoff (sum of all components).
- **Dotted gray line** shows the actual payoff of the target position.

If the synthetic works correctly, the black and gray lines will be perfectly superimposed.

> **Key Concept:** These are payoff diagrams *at expiry*. The synthetics replicate the target at expiry by construction (that is what parity guarantees). They also replicate the target *before* expiry, but that is harder to show visually since it depends on option pricing models.

In [ ]:
S_T = np.linspace(50, 150, 300)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Synthetic Long Stock: long call + short put + bond
ax = axes[0, 0]
call_payoff = np.maximum(S_T - K, 0)
put_payoff = -np.maximum(K - S_T, 0)
bond_payoff = K * np.ones_like(S_T)
synthetic_stock = call_payoff + put_payoff + bond_payoff

ax.plot(S_T, call_payoff, '--', color=PRIMARY, alpha=0.6, label='Long Call')
ax.plot(S_T, put_payoff, '--', color=SECONDARY, alpha=0.6, label='Short Put')
ax.plot(S_T, bond_payoff, '--', color=ACCENT, alpha=0.6, label=f'Bond (K={K})')
ax.plot(S_T, synthetic_stock, 'k-', linewidth=2.5, label='Synthetic = Stock')
ax.plot(S_T, S_T, ':', color='gray', linewidth=1, label='Actual Stock')
ax.set_xlabel('$S_T$')
ax.set_ylabel('Payoff')
ax.set_title('Synthetic Long Stock')
ax.legend(fontsize=8)

# 2. Synthetic Short Stock: short call + long put + borrow
ax = axes[0, 1]
short_call = -np.maximum(S_T - K, 0)
long_put = np.maximum(K - S_T, 0)
borrow = -K * np.ones_like(S_T)
synthetic_short = short_call + long_put + borrow

ax.plot(S_T, short_call, '--', color=PRIMARY, alpha=0.6, label='Short Call')
ax.plot(S_T, long_put, '--', color=SECONDARY, alpha=0.6, label='Long Put')
ax.plot(S_T, borrow, '--', color=ACCENT, alpha=0.6, label=f'Borrow (-K)')
ax.plot(S_T, synthetic_short, 'k-', linewidth=2.5, label='Synthetic = -Stock')
ax.plot(S_T, -S_T, ':', color='gray', linewidth=1, label='Actual Short Stock')
ax.set_xlabel('$S_T$')
ax.set_ylabel('Payoff')
ax.set_title('Synthetic Short Stock')
ax.legend(fontsize=8)

# 3. Synthetic Long Call: long put + long stock - bond
ax = axes[1, 0]
long_put2 = np.maximum(K - S_T, 0)
long_stock = S_T
short_bond = -K * np.ones_like(S_T)
synthetic_call = long_put2 + long_stock + short_bond

ax.plot(S_T, long_put2, '--', color=SECONDARY, alpha=0.6, label='Long Put')
ax.plot(S_T, long_stock, '--', color=TERTIARY, alpha=0.6, label='Long Stock')
ax.plot(S_T, short_bond, '--', color=ACCENT, alpha=0.6, label=f'Short Bond (-K)')
ax.plot(S_T, synthetic_call, 'k-', linewidth=2.5, label='Synthetic = Call')
ax.plot(S_T, np.maximum(S_T - K, 0), ':', color='gray', linewidth=1, label='Actual Call')
ax.set_xlabel('$S_T$')
ax.set_ylabel('Payoff')
ax.set_title('Synthetic Long Call')
ax.legend(fontsize=8)

# 4. Synthetic Long Put: long call + short stock + bond
ax = axes[1, 1]
long_call = np.maximum(S_T - K, 0)
short_stock = -S_T
long_bond = K * np.ones_like(S_T)
synthetic_put = long_call + short_stock + long_bond

ax.plot(S_T, long_call, '--', color=PRIMARY, alpha=0.6, label='Long Call')
ax.plot(S_T, short_stock, '--', color=TERTIARY, alpha=0.6, label='Short Stock')
ax.plot(S_T, long_bond, '--', color=ACCENT, alpha=0.6, label=f'Bond (+K)')
ax.plot(S_T, synthetic_put, 'k-', linewidth=2.5, label='Synthetic = Put')
ax.plot(S_T, np.maximum(K - S_T, 0), ':', color='gray', linewidth=1, label='Actual Put')
ax.set_xlabel('$S_T$')
ax.set_ylabel('Payoff')
ax.set_title('Synthetic Long Put')
ax.legend(fontsize=8)

plt.suptitle('Synthetic Positions from Put-Call Parity', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Reading the Synthetic Position Plots

All four panels confirm that the synthetic constructions work perfectly:

**Top-left: Synthetic Long Stock.** The call (blue dashed, kinked upward at $K$) plus the short put (coral dashed, kinked downward at $K$) plus the bond (gold, flat at $K$) combine to produce a straight line equal to $S_T$ -- identical to owning the stock.

**Top-right: Synthetic Short Stock.** The reverse construction produces $-S_T$, matching an actual short stock position.

**Bottom-left: Synthetic Long Call.** A long put (coral dashed) + long stock (green dashed) + short bond (gold dashed at $-K$) combine to produce the familiar hockey-stick call payoff $\max(S_T - K, 0)$.

**Bottom-right: Synthetic Long Put.** A long call + short stock + long bond combine to produce the put payoff $\max(K - S_T, 0)$.

In each panel, the thick black line (synthetic) is perfectly superimposed on the dotted gray line (actual). The synthetic is a perfect replica.

> **Key Concept:** These four diagrams are the visual "proof" that put-call parity works. Each synthetic position exactly replicates its target because the parity equation is an identity -- it holds for every possible value of $S_T$.

> **CFA Exam Tip:** When asked to "create a synthetic X," just rearrange parity to isolate X on one side. Then interpret the other side as a trading strategy. Long = buy, short = sell, $+Ke^{-rT}$ = lend (buy bond), $-Ke^{-rT}$ = borrow (sell bond).

> **Common Mistake:** Confusing the direction of the bond. In a synthetic long stock ($S = C - P + Ke^{-rT}$), the $+Ke^{-rT}$ means you *lend* (buy a bond). In a synthetic long call ($C = S + P - Ke^{-rT}$), the $-Ke^{-rT}$ means you *borrow*. Getting the sign wrong flips the entire strategy.### Why Synthetics Matter in Practice

| Use case | What you do | Why |
|:---------|:-----------|:----|
| **Short-selling ban** | Synthetic short via options | Regulatory restrictions in some markets |
| **Tighter spreads** | Construct position via more liquid instruments | Options market may be more liquid than stock for some names |
| **Box spread arbitrage** | Combine synthetic long and short at different strikes | Locks in a risk-free rate; used to detect funding rate discrepancies |
| **Hedging** | Convert between positions cheaply | Change exposure profile without unwinding existing trades |

> **Key Concept:** The ability to replicate any position synthetically means that mispricing in ANY one of the four instruments (call, put, stock, bond) can be exploited using the other three. This is what keeps option markets efficient.


---

## 10. Summary and Key Takeaways

### What We Covered

1. **Put-call parity** for European options: $C - P = S - Ke^{-rT}$. Derived from a pure no-arbitrage argument using replicating portfolios. No distributional assumptions needed.

2. **Numerical verification** across 18 parameter combinations confirmed parity holds to machine precision ($\sim 10^{-14}$).

3. **Visual verification** showed both sides of the parity equation as perfectly overlapping lines.

4. **Arbitrage strategies** for both directions of parity violation, with complete cash flow tables showing zero risk at expiry.

5. **No-arbitrage bounds** on option prices: $C \geq \max(0, S - Ke^{-rT})$ and $P \geq \max(0, Ke^{-rT} - S)$.

6. **American options**: parity becomes bounds $S - K \leq C_A - P_A \leq S - Ke^{-rT}$ due to the early exercise premium on puts.

7. **Dividend adjustments**: replace $S$ with $S - D_{\text{PV}}$ in all formulas.

8. **Synthetic positions**: any instrument can be replicated from the other three.

### The Big Picture

Put-call parity is the cornerstone of options theory because it:
- Provides a **model-free** relationship (unlike Black-Scholes, which assumes GBM).
- Enables **arbitrage detection** when market prices deviate.
- Allows **synthetic replication** of any instrument from the other three.
- Serves as a **sanity check** for any pricing model.

If you remember only one equation from options theory, make it this one:

$$\boxed{C + Ke^{-rT} = P + S}$$

---

## References

1. Stoll, H. R. (1969). *The relationship between put and call option prices*. Journal of Finance, 24(5), 801-824.
2. Merton, R. C. (1973). *Theory of rational option pricing*. Bell Journal of Economics and Management Science, 4(1), 141-183.
3. Hull, J. C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
4. Cox, J., & Rubinstein, M. (1985). *Options Markets*. Prentice-Hall.
5. Shreve, S. E. (2004). *Stochastic Calculus for Finance I: The Binomial Asset Pricing Model*. Springer.### Formula Reference Card

| Relationship | Formula | Conditions |
|:---|:---|:---|
| European parity | $C - P = S - Ke^{-rT}$ | No dividends |
| Parity with dividends | $C - P = S - PV(D) - Ke^{-rT}$ | Known discrete dividends |
| Parity with yield | $C - P = Se^{-qT} - Ke^{-rT}$ | Continuous dividend yield |
| American bounds | $S - K \leq C_A - P_A \leq S - Ke^{-rT}$ | Non-dividend-paying |
| Call lower bound | $C \geq \max(0, S - Ke^{-rT})$ | European |
| Put lower bound | $P \geq \max(0, Ke^{-rT} - S)$ | European |


> **Final note:** Mastery of the concepts in this notebook is essential for the CFA Level 1 exam, as well as for practical financial analysis work. Practice the worked examples by hand and verify your understanding by reproducing the code from scratch.
